# Healthcare Provider Fraud Detection — Reproducibility Audit (2026-11-18)

**Purpose:** Re-run the originally submitted provider-level benchmark using the supplied Kaggle training data, without treating its labels as proven fraud adjudications or claiming production deployment.

**Evidence caution:** This is a post-filing reproducibility audit. It is not a claim that these newly generated outputs existed at the petition filing date. Provider-level historical labels do **not** validate claim-level prepayment detection.

Run this notebook from the `NIW_Project1` directory. The four original `Train*.csv` files should be in `data/`.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
assert all((DATA_DIR / f).exists() for f in ["Train-1542865627584.csv","Train_Beneficiarydata-1542865627584.csv","Train_Inpatientdata-1542865627584.csv","Train_Outpatientdata-1542865627584.csv"]), "Missing required training files"
print("Dataset folder:", DATA_DIR)

Dataset folder: /mnt/data/NIW_Project1/data


## Feature engineering — original logic, portable paths

In [2]:
"""
Provider-Level Healthcare Claims Fraud Detection
Step 1: Feature Engineering

Builds provider-level aggregate features from the CMS-style Inpatient, Outpatient,
and Beneficiary claims tables (Kaggle "Healthcare Provider Fraud Detection Analysis"
dataset). The prediction target (PotentialFraud) is defined at the PROVIDER level,
not the individual claim level, so all features must be aggregated up to one row
per provider before modeling.
"""

import pandas as pd
import numpy as np

DATA_DIR = str(DATA_DIR)

# ---------------------------------------------------------------------------
# 1. Load raw tables
# ---------------------------------------------------------------------------
labels = pd.read_csv(f"{DATA_DIR}/Train-1542865627584.csv")
beneficiary = pd.read_csv(f"{DATA_DIR}/Train_Beneficiarydata-1542865627584.csv")
inpatient = pd.read_csv(f"{DATA_DIR}/Train_Inpatientdata-1542865627584.csv")
outpatient = pd.read_csv(f"{DATA_DIR}/Train_Outpatientdata-1542865627584.csv")

print(f"Providers (labels):   {labels.shape}")
print(f"Beneficiary records:  {beneficiary.shape}")
print(f"Inpatient claims:     {inpatient.shape}")
print(f"Outpatient claims:    {outpatient.shape}")

# ---------------------------------------------------------------------------
# 2. Combine inpatient + outpatient into one claims table
#    (mark claim type so we can compute the inpatient/outpatient mix per provider)
# ---------------------------------------------------------------------------
inpatient = inpatient.copy()
outpatient = outpatient.copy()
inpatient["ClaimType"] = "Inpatient"
outpatient["ClaimType"] = "Outpatient"

diag_cols = [f"ClmDiagnosisCode_{i}" for i in range(1, 11)]
proc_cols_ip = [f"ClmProcedureCode_{i}" for i in range(1, 7)]

common_cols = [
    "BeneID", "ClaimID", "ClaimStartDt", "ClaimEndDt", "Provider",
    "InscClaimAmtReimbursed", "AttendingPhysician", "OperatingPhysician",
    "OtherPhysician", "DeductibleAmtPaid", "ClmAdmitDiagnosisCode", "ClaimType"
] + diag_cols

claims = pd.concat(
    [
        inpatient[common_cols + proc_cols_ip + ["AdmissionDt", "DischargeDt"]],
        outpatient[common_cols].assign(
            **{c: np.nan for c in proc_cols_ip}, AdmissionDt=np.nan, DischargeDt=np.nan
        ),
    ],
    ignore_index=True,
)

claims["ClaimStartDt"] = pd.to_datetime(claims["ClaimStartDt"])
claims["ClaimEndDt"] = pd.to_datetime(claims["ClaimEndDt"])
claims["AdmissionDt"] = pd.to_datetime(claims["AdmissionDt"])
claims["DischargeDt"] = pd.to_datetime(claims["DischargeDt"])
claims["ClaimDurationDays"] = (claims["ClaimEndDt"] - claims["ClaimStartDt"]).dt.days
claims["LengthOfStayDays"] = (claims["DischargeDt"] - claims["AdmissionDt"]).dt.days

# Diagnosis / procedure code counts per claim (how many distinct codes billed)
claims["NumDiagCodes"] = claims[diag_cols].notna().sum(axis=1)
claims["NumProcCodes"] = claims[proc_cols_ip].notna().sum(axis=1)

# Physician-role overlap flag: same physician billed in more than one role on
# the same claim. This is a documented red flag in CMS program-integrity
# guidance (self-referral / role-stacking patterns).
claims["PhysicianRoleOverlap"] = (
    ((claims["AttendingPhysician"] == claims["OperatingPhysician"]) & claims["AttendingPhysician"].notna())
    | ((claims["AttendingPhysician"] == claims["OtherPhysician"]) & claims["AttendingPhysician"].notna())
    | ((claims["OperatingPhysician"] == claims["OtherPhysician"]) & claims["OperatingPhysician"].notna())
).astype(int)

print(f"\nCombined claims table: {claims.shape}")

# ---------------------------------------------------------------------------
# 3. Beneficiary-level features (age, chronic conditions) joined onto claims
# ---------------------------------------------------------------------------
beneficiary = beneficiary.copy()
beneficiary["DOB"] = pd.to_datetime(beneficiary["DOB"])
beneficiary["DOD"] = pd.to_datetime(beneficiary["DOD"])
# Approximate age as of the dataset's reference year (2009, per the Kaggle
# dataset documentation) since DOD is only populated for deceased beneficiaries.
beneficiary["Age"] = 2009 - beneficiary["DOB"].dt.year

chronic_cols = [c for c in beneficiary.columns if c.startswith("ChronicCond_")]
# Chronic condition flags are coded 1 = yes, 2 = no in this dataset; convert to 0/1
for c in chronic_cols:
    beneficiary[c] = (beneficiary[c] == 1).astype(int)
beneficiary["ChronicConditionCount"] = beneficiary[chronic_cols].sum(axis=1)
beneficiary["IsDeceased"] = beneficiary["DOD"].notna().astype(int)

bene_features = beneficiary[
    ["BeneID", "Age", "ChronicConditionCount", "IsDeceased", "RenalDiseaseIndicator", "State"]
].copy()
bene_features["RenalDiseaseIndicator"] = (bene_features["RenalDiseaseIndicator"] == "Y").astype(int)

claims = claims.merge(bene_features, on="BeneID", how="left")

# ---------------------------------------------------------------------------
# 4. Aggregate claims up to one row per PROVIDER (the label's grain)
# ---------------------------------------------------------------------------
agg = claims.groupby("Provider").agg(
    NumClaims=("ClaimID", "count"),
    NumUniqueBeneficiaries=("BeneID", "nunique"),
    TotalReimbursed=("InscClaimAmtReimbursed", "sum"),
    AvgReimbursedPerClaim=("InscClaimAmtReimbursed", "mean"),
    StdReimbursedPerClaim=("InscClaimAmtReimbursed", "std"),
    MaxReimbursedClaim=("InscClaimAmtReimbursed", "max"),
    TotalDeductible=("DeductibleAmtPaid", "sum"),
    AvgClaimDurationDays=("ClaimDurationDays", "mean"),
    AvgLengthOfStayDays=("LengthOfStayDays", "mean"),
    AvgNumDiagCodes=("NumDiagCodes", "mean"),
    AvgNumProcCodes=("NumProcCodes", "mean"),
    PhysicianRoleOverlapRate=("PhysicianRoleOverlap", "mean"),
    NumUniqueAttendingPhysicians=("AttendingPhysician", "nunique"),
    NumUniqueOperatingPhysicians=("OperatingPhysician", "nunique"),
    PctInpatient=("ClaimType", lambda x: (x == "Inpatient").mean()),
    AvgBeneficiaryAge=("Age", "mean"),
    AvgChronicConditions=("ChronicConditionCount", "mean"),
    PctDeceasedBeneficiaries=("IsDeceased", "mean"),
    PctRenalDisease=("RenalDiseaseIndicator", "mean"),
    NumUniqueStates=("State", "nunique"),
).reset_index()

# Claims-per-beneficiary concentration: high values suggest a small pool of
# beneficiaries being billed repeatedly, a documented pattern in provider-level
# fraud rings.
agg["ClaimsPerBeneficiary"] = agg["NumClaims"] / agg["NumUniqueBeneficiaries"]

# Fill claim-type-specific NaNs (providers with zero inpatient claims have no
# length-of-stay to average) with 0 rather than dropping the provider.
agg["AvgLengthOfStayDays"] = agg["AvgLengthOfStayDays"].fillna(0)
agg["StdReimbursedPerClaim"] = agg["StdReimbursedPerClaim"].fillna(0)

# ---------------------------------------------------------------------------
# 5. Attach the fraud label
# ---------------------------------------------------------------------------
dataset = agg.merge(labels, on="Provider", how="inner")
dataset["Label"] = (dataset["PotentialFraud"] == "Yes").astype(int)

print(f"\nFinal provider-level dataset: {dataset.shape}")
print(f"Fraud rate: {dataset['Label'].mean():.4f}")

dataset.to_csv(str(OUT / "provider_level_features.csv"), index=False)
print("\nSaved to provider_level_features.csv")
print("\nFeature columns:")
print([c for c in dataset.columns if c not in ("Provider", "PotentialFraud", "Label")])


Providers (labels):   (5410, 2)
Beneficiary records:  (138556, 25)
Inpatient claims:     (40474, 30)
Outpatient claims:    (517737, 27)



Combined claims table: (558211, 35)



Final provider-level dataset: (5410, 24)
Fraud rate: 0.0935

Saved to provider_level_features.csv

Feature columns:
['NumClaims', 'NumUniqueBeneficiaries', 'TotalReimbursed', 'AvgReimbursedPerClaim', 'StdReimbursedPerClaim', 'MaxReimbursedClaim', 'TotalDeductible', 'AvgClaimDurationDays', 'AvgLengthOfStayDays', 'AvgNumDiagCodes', 'AvgNumProcCodes', 'PhysicianRoleOverlapRate', 'NumUniqueAttendingPhysicians', 'NumUniqueOperatingPhysicians', 'PctInpatient', 'AvgBeneficiaryAge', 'AvgChronicConditions', 'PctDeceasedBeneficiaries', 'PctRenalDisease', 'NumUniqueStates', 'ClaimsPerBeneficiary']


## Model training — original logic, portable paths

In [3]:
"""
Provider-Level Healthcare Claims Fraud Detection
Step 2: Baseline and Primary Model Training + Evaluation

Trains a simple Logistic Regression baseline and a Gradient Boosted Trees primary
model on the provider-level features built in step 1. Reports the metrics needed
to substantiate an NIW petition's technical claims: precision, recall, ROC-AUC,
PR-AUC, and a false-positive-rate comparison between the two models AT A MATCHED
RECALL LEVEL, so the comparison is apples-to-apples rather than an arbitrary
threshold pick.

IMPORTANT: All numbers printed by this script are real, computed from the actual
data. Nothing here is a placeholder or an assumed result.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, confusion_matrix, classification_report
)
import json

RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# 1. Load features and split by PROVIDER (the label's grain) into train/test
# ---------------------------------------------------------------------------
df = pd.read_csv(str(OUT / "provider_level_features.csv"))

feature_cols = [c for c in df.columns if c not in ("Provider", "PotentialFraud", "Label")]
X = df[feature_cols].fillna(0)
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set: {X_train.shape[0]} providers ({y_train.mean():.4f} fraud rate)")
print(f"Test set:  {X_test.shape[0]} providers ({y_test.mean():.4f} fraud rate)")

# ---------------------------------------------------------------------------
# 2. Baseline model: Logistic Regression on standardized features
#    (This is the "rule-based / simple baseline" referenced in the petition.)
# ---------------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
baseline.fit(X_train_scaled, y_train)
baseline_proba = baseline.predict_proba(X_test_scaled)[:, 1]

# ---------------------------------------------------------------------------
# 3. Primary model: Gradient Boosted Trees
#    (scikit-learn's GradientBoostingClassifier — xgboost was unavailable in
#    this sandboxed environment with no internet access; this is a standard,
#    well-established substitute using the same underlying boosted-tree family.)
# ---------------------------------------------------------------------------
primary = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE
)
primary.fit(X_train, y_train)
primary_proba = primary.predict_proba(X_test)[:, 1]

# Also fit a Random Forest for comparison / robustness check
rf = RandomForestClassifier(
    n_estimators=300, max_depth=6, class_weight="balanced", random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------------------------
# 4. Evaluation: ROC-AUC, PR-AUC (average precision)
# ---------------------------------------------------------------------------
results = {}
for name, proba in [("Baseline (Logistic Regression)", baseline_proba),
                     ("Primary (Gradient Boosted Trees)", primary_proba),
                     ("Random Forest (robustness check)", rf_proba)]:
    auc = roc_auc_score(y_test, proba)
    ap = average_precision_score(y_test, proba)
    results[name] = {"ROC_AUC": auc, "PR_AUC": ap}
    print(f"\n{name}")
    print(f"  ROC-AUC: {auc:.4f}")
    print(f"  PR-AUC (Average Precision): {ap:.4f}")

# ---------------------------------------------------------------------------
# 5. False-positive-rate comparison AT A MATCHED RECALL LEVEL
#    This directly answers the petition's open question: "30% compared with
#    what, at what recall?" — here we fix recall and report the actual FPR.
# ---------------------------------------------------------------------------
def fpr_at_recall(y_true, y_proba, target_recall):
    """Find the false positive rate at the threshold achieving >= target_recall."""
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    # find smallest threshold (i.e. lowest index where tpr first reaches target)
    idx = np.searchsorted(tpr, target_recall)
    idx = min(idx, len(tpr) - 1)
    return fpr[idx], tpr[idx], thresholds[idx]

print("\n" + "=" * 70)
print("FALSE POSITIVE RATE COMPARISON AT MATCHED RECALL (target recall = 0.70)")
print("=" * 70)

target_recall = 0.70
baseline_fpr, baseline_actual_recall, baseline_thresh = fpr_at_recall(y_test, baseline_proba, target_recall)
primary_fpr, primary_actual_recall, primary_thresh = fpr_at_recall(y_test, primary_proba, target_recall)

print(f"\nBaseline (Logistic Regression):")
print(f"  At recall = {baseline_actual_recall:.3f}, false positive rate = {baseline_fpr:.4f} ({baseline_fpr*100:.2f}%)")
print(f"  Threshold: {baseline_thresh:.4f}")

print(f"\nPrimary (Gradient Boosted Trees):")
print(f"  At recall = {primary_actual_recall:.3f}, false positive rate = {primary_fpr:.4f} ({primary_fpr*100:.2f}%)")
print(f"  Threshold: {primary_thresh:.4f}")

if baseline_fpr > 0:
    relative_reduction = (baseline_fpr - primary_fpr) / baseline_fpr * 100
    pct_point_reduction = (baseline_fpr - primary_fpr) * 100
    print(f"\n>>> Relative FPR reduction: {relative_reduction:.1f}%")
    print(f">>> Percentage-point FPR reduction: {pct_point_reduction:.2f} percentage points")
else:
    relative_reduction = None
    pct_point_reduction = (baseline_fpr - primary_fpr) * 100

# ---------------------------------------------------------------------------
# 6. Confusion matrix and classification report at the matched-recall threshold
# ---------------------------------------------------------------------------
primary_preds_at_thresh = (primary_proba >= primary_thresh).astype(int)
cm = confusion_matrix(y_test, primary_preds_at_thresh)
print(f"\nPrimary model confusion matrix at recall={primary_actual_recall:.3f} threshold:")
print(f"                 Predicted No-Fraud   Predicted Fraud")
print(f"Actual No-Fraud       {cm[0][0]:>6}              {cm[0][1]:>6}")
print(f"Actual Fraud          {cm[1][0]:>6}              {cm[1][1]:>6}")

print("\nFull classification report (primary model, 0.5 threshold):")
print(classification_report(y_test, (primary_proba >= 0.5).astype(int), target_names=["No Fraud", "Fraud"]))

# ---------------------------------------------------------------------------
# 7. Save everything needed for reporting + downstream explainability script
# ---------------------------------------------------------------------------
summary = {
    "n_providers_total": int(len(df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "fraud_rate_overall": float(y.mean()),
    "baseline_roc_auc": float(results["Baseline (Logistic Regression)"]["ROC_AUC"]),
    "baseline_pr_auc": float(results["Baseline (Logistic Regression)"]["PR_AUC"]),
    "primary_roc_auc": float(results["Primary (Gradient Boosted Trees)"]["ROC_AUC"]),
    "primary_pr_auc": float(results["Primary (Gradient Boosted Trees)"]["PR_AUC"]),
    "rf_roc_auc": float(results["Random Forest (robustness check)"]["ROC_AUC"]),
    "rf_pr_auc": float(results["Random Forest (robustness check)"]["PR_AUC"]),
    "target_recall": target_recall,
    "baseline_fpr_at_target_recall": float(baseline_fpr),
    "baseline_actual_recall": float(baseline_actual_recall),
    "primary_fpr_at_target_recall": float(primary_fpr),
    "primary_actual_recall": float(primary_actual_recall),
    "relative_fpr_reduction_pct": float(relative_reduction) if relative_reduction is not None else None,
    "pct_point_fpr_reduction": float(pct_point_reduction),
}

with open(str(OUT / "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# Save test set + predictions for the explainability script
X_test.assign(Label=y_test.values, PrimaryModelProba=primary_proba).to_csv(
    str(OUT / "test_predictions.csv"), index=False
)

import pickle
with open(str(OUT / "primary_model.pkl"), "wb") as f:
    pickle.dump(primary, f)
X_train.to_csv(str(OUT / "X_train.csv"), index=False)

print("\nSaved: results_summary.json, test_predictions.csv, primary_model.pkl, X_train.csv")


Train set: 4057 providers (0.0934 fraud rate)
Test set:  1353 providers (0.0939 fraud rate)



Baseline (Logistic Regression)
  ROC-AUC: 0.9510
  PR-AUC (Average Precision): 0.7260

Primary (Gradient Boosted Trees)
  ROC-AUC: 0.9551
  PR-AUC (Average Precision): 0.7331

Random Forest (robustness check)
  ROC-AUC: 0.9485
  PR-AUC (Average Precision): 0.6764

FALSE POSITIVE RATE COMPARISON AT MATCHED RECALL (target recall = 0.70)

Baseline (Logistic Regression):
  At recall = 0.709, false positive rate = 0.0604 (6.04%)
  Threshold: 0.7561

Primary (Gradient Boosted Trees):
  At recall = 0.701, false positive rate = 0.0449 (4.49%)
  Threshold: 0.3442

>>> Relative FPR reduction: 25.7%
>>> Percentage-point FPR reduction: 1.55 percentage points

Primary model confusion matrix at recall=0.701 threshold:
                 Predicted No-Fraud   Predicted Fraud
Actual No-Fraud         1171                  55
Actual Fraud              38                  89

Full classification report (primary model, 0.5 threshold):
              precision    recall  f1-score   support

    No Fraud      

## Audit caveats and Power BI exports

- Thresholds in the original script are selected using the held-out test labels. This **leaks evaluation information** into threshold choice; the reported matched-recall FPR is exploratory and not an unbiased prospective estimate. A later corrected workflow must choose thresholds using a validation set, then evaluate once on untouched test providers.
- This is provider-level prediction based on aggregated historical claims, not individual-claim prepayment adjudication.
- `PotentialFraud` is a dataset label, not independent proof of criminal fraud.
- The original script uses `fillna(0)` for all remaining missing values; investigate missingness and alternative imputations.
- The physician-role-overlap flag is an exploratory signal, not independently verified evidence of improper referral or fraud.
- Raw feature deviations from training means are **not** causal explanations or SHAP values.
- Do not publish sensitive identifiers in public dashboards; use pseudonymous IDs and aggregated data.


In [4]:
from sklearn.metrics import confusion_matrix
summary = json.loads((OUT / "results_summary.json").read_text())
print(pd.Series(summary).to_string())
# Provider-level test predictions exported by original script (no provider IDs).
pred = pd.read_csv(OUT / "test_predictions.csv")
print("Prediction export:", pred.shape)
print("Missing model scores:", pred.PrimaryModelProba.isna().sum())
# Aggregated Power BI metrics (safe to share as benchmark summaries)
metric_keys = ["baseline_roc_auc","baseline_pr_auc","primary_roc_auc","primary_pr_auc","rf_roc_auc","rf_pr_auc","baseline_fpr_at_target_recall","primary_fpr_at_target_recall","relative_fpr_reduction_pct"]
pd.DataFrame([{"metric":k,"value":summary.get(k),"context":"historical provider-level Kaggle benchmark; original test-set-selected thresholds"} for k in metric_keys]).to_csv(OUT / "powerbi_model_metrics.csv",index=False)
# Binned risk distribution, without releasing row-level claims or beneficiary data
pred["risk_band"] = pd.cut(pred.PrimaryModelProba, bins=[-0.001,.2,.4,.6,.8,1.001], labels=["0–20%","20–40%","40–60%","60–80%","80–100%"] )
pred.groupby(["risk_band","Label"],observed=True).size().rename("providers").reset_index().to_csv(OUT / "powerbi_risk_bands.csv",index=False)
print("Power BI exports saved.")

n_providers_total                5410.000000
n_train                          4057.000000
n_test                           1353.000000
fraud_rate_overall                  0.093530
baseline_roc_auc                    0.950964
baseline_pr_auc                     0.726033
primary_roc_auc                     0.955097
primary_pr_auc                      0.733084
rf_roc_auc                          0.948485
rf_pr_auc                           0.676374
target_recall                       0.700000
baseline_fpr_at_target_recall       0.060359
baseline_actual_recall              0.708661
primary_fpr_at_target_recall        0.044861
primary_actual_recall               0.700787
relative_fpr_reduction_pct         25.675676
pct_point_fpr_reduction             1.549755
Prediction export: (1353, 23)
Missing model scores: 0
Power BI exports saved.
